In [ ]:
!pip install -r requirements.txt

In [ ]:
# Upgrade protobuf before any imports to resolve version conflict with tensorflow
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'protobuf>=6.31.1'])
#import packages
import tqdm
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook
import math

import numpy as np
from numpy import array

from numpy.linalg import norm
import pickle
import pandas as pd
import torch

import math
import sys
import snowflake.snowpark.modin.plugin
import os
import snowflake.snowpark.functions as F

In [ ]:
%run ./PY_INITIALIZE.ipynb

In [ ]:
from sklearn import tree
import xgboost as xgb
from sklearn.metrics import accuracy_score
#import sqlalchemy

In [ ]:
import snowflake.snowpark.modin.plugin
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import modin.pandas as pd

In [ ]:
clmsptData = pd.read_snowflake(f"{database_name}.{schema_name}.FINAL_TABLE")
logger.info(f"The clmsptData is read into a dataframe")

In [ ]:
clmsptData['INC_FROM'] = clmsptData['INC_FROM'].astype(str)
clmsptData['clm_yr'] = clmsptData['INC_FROM'].str[:4]
clmsptData['clm_yr'] = clmsptData['clm_yr'].astype(int)
strt_yr = clmsptData['clm_yr'].min()
last_yr = clmsptData['clm_yr'].max()
clmsptData.drop(columns=['INC_FROM','INC_TO'], inplace=True)

logger.info(f"Extracted the first ({strt_yr}) and last ({last_yr}) claim year")

In [ ]:
def test_dataprep(LstClmYr):
    df = clmsptData[clmsptData['clm_yr']==LstClmYr]
    print(list(df.columns.values))
    colsorder = ["MBR_ID", 'FST_NAME', 'MDL_NAME', 'LST_NAME', 'PT_AGE', 'PT_ZIP',\
    'ICD_CD','CPT_CD','ICDCD_NUMCODED', 'OMC_ICD_RISK_NBR','SUBCD_NBR',\
        'PAID_AMT', 'clm_yr']
    df = df[colsorder]
    
    df.drop_duplicates(subset=['MBR_ID', 'ICDCD_NUMCODED', 'SUBCD_NBR'], inplace=True)
    df['OMC_ICD_RISK_NBR'] = df['OMC_ICD_RISK_NBR'].astype(int)
    df.to_snowflake(f"{database_name}.{schema_name}.Testing_Table", if_exists='replace', index=False)
    logging.info(f"Testing data file written to database: {database_name} schema: {schema_name}")


In [ ]:
def extract_test_data(df):
    print(list(df.columns.values))
    df['MBR_ID'] = df['MBR_ID'].astype(str)
    df['FST_NAME'] = df['FST_NAME'].astype(str)
    df['MDL_NAME'] = df['MDL_NAME'].astype(str)
    df['LST_NAME'] = df['LST_NAME'].astype(str)
    df['OMC_ICD_RISK_NBR'] = df['OMC_ICD_RISK_NBR'].astype(int)
    df['ICDCD_NUMCODED'] = df['ICDCD_NUMCODED'].astype(float)
    df['SUBCD_NBR'] = df['SUBCD_NBR'].astype(float)
    df['PT_AGE'] = df['PT_AGE'].astype(int)
    df['PT_ZIP'] = df['PT_ZIP'].astype(float)
    logger.info(f"The clmsptData is read into a dataframe")
    testData = df._to_pandas()
    print(list(testData.columns.values))

In [ ]:

def get_x_tst_data(prepped_data):
    if(prepped_data.empty): 
        print("Dataframe for the claim difference range "+level+" is empty")
        null=1
        # Read the training data from the sf_2016_2017 dataframe
        x1 = {}
        x2 = {}
        x3 = {}
        x4 = {}
        x5 = {}
    else:
        # Read the training data from the sf_2016_2017 dataframe
        x1 = np.vstack(prepped_data['PT_AGE']).astype(int)
        x2 = np.vstack(prepped_data['PT_ZIP']).astype(int)
        x3 = np.vstack(prepped_data['ICDCD_NUMCODED']).astype(np.float32)
        x4 = np.vstack(prepped_data['OMC_ICD_RISK_NBR']).astype(np.float32)
        x5 = np.vstack(prepped_data['SUBCD_NBR']).astype(np.float32)
    return x1, x2, x3,x4,x5  

In [ ]:
def get_tensor_dataset(x1,x2, x3, x4, x5):
    # COnverting training data and testing data to torch tensor
    x_train = np.column_stack((x1,x2, x3, x4, x5))
    #y_train = nxt_yr_diff
    test_x = torch.tensor(x_train, dtype=torch.float)
    print(test_x.shape)
    print(test_x.shape[-1])
    
    return test_x

In [ ]:
from snowflake.ml.registry import Registry
def retrieve_model(model_name):
    #Initialize the model list dataframe
    df_models = []
    filtered_models = []
    reg = Registry(
        session=session, 
        database_name=database_name, 
        schema_name=model_schema
    )
    df_models = reg.show_models()
    if len(df_models)>0:
        filtered_models = df_models[df_models['name'].str.contains(model_name, case=False, na=False)]

    if (len(filtered_models)>0):       
        models_sort = filtered_models.sort_values(by="created_on", ascending=False)
        last2 = models_sort.iloc[0]["name"][-2:]
        model_name = models_sort.iloc[0]["name"]
        model_version = "v"+str(last2)
        model = reg.get_model(model_name).version(model_version)

        
        return model
    else:
        print("Trained model not available")
        return null
    

In [ ]:
def load_table(StoredTable, CleanedTable):
        
        sql_query = f"""CREATE or REPLACE TABLE {CleanedTable} AS
            SELECT 
            TO_VARIANT(MBR_ID) AS MBR_ID,
            TO_VARIANT(FST_NAME) AS FST_NAME,
            TO_VARIANT(MDL_NAME) AS MDL_NAME,
            TO_VARIANT(LST_NAME) AS LST_NAME,
            TO_VARIANT("PT_AGE") AS PT_AGE,
            TO_VARIANT(PT_ZIP) AS PT_ZIP,
            TO_VARIANT(ICD_CD) AS ICD_CD,
            TO_VARIANT(CPT_CD) AS CPT_CD,
            TO_VARIANT("ICDCD_NUMCODED") AS ICDCD_NUMCODED,
            TO_VARIANT("SUBCD_NBR") AS SUBCD_NBR,
            TO_VARIANT(OMC_ICD_RISK_NBR) AS OMC_ICD_RISK_NBR,
            TO_VARIANT(PAID_AMT) AS PAID_AMT,
            TO_VARIANT("clm_yr") AS CLM_YR
            FROM {StoredTable}"""
        
        df = session.sql(sql_query)
        df.collect()
        
        testData_pd = pd.read_snowflake(f"{database_name}.{schema_name}.{CleanedTable}")
        testData_pd.fillna(0, inplace=True)
        return testData_pd

In [ ]:
import snowflake.snowpark.modin.plugin
import snowflake.snowpark.functions as F
from snowflake.ml.model import model_signature
from snowflake.ml.model.model_signature import ModelSignature
from snowflake.ml.model.model_signature import FeatureSpec
from snowflake.ml.model.model_signature import DataType
from snowflake.snowpark.functions import col, lit, typeof
from snowflake.snowpark.types import IntegerType, StringType

def test():
    results= session.sql(f"SHOW TABLES LIKE 'TESTING_TABLE' IN {database_name}.{schema_name}").collect()
            
    if len(results)>0:
        # file exists
        print("Testing data table exists")
        #testData = pd.read_snowflake(f"{database_name}.{schema_name}.VALIDATING_TABLE")
    else:
        print("Testing dataprep needs to be done")  
        test_dataprep(int(last_yr))
    TableName = "TESTING_TABLE"
    NewTableName = "TESTING_DATA"
    """Load the testing data from the default database, schema """
    testData = load_table(TableName, NewTableName)
    
    testData['MBR_ID'] = testData['MBR_ID'].astype(str)
    testData['FST_NAME'] = testData['FST_NAME'].astype(str)
    testData['MDL_NAME'] = testData['MDL_NAME'].astype(str)
    testData['LST_NAME'] = testData['LST_NAME'].astype(str)
    testData['OMC_ICD_RISK_NBR'] = testData['OMC_ICD_RISK_NBR'].astype(int)
    testData['ICDCD_NUMCODED'] = testData['ICDCD_NUMCODED'].astype(float)
    testData['SUBCD_NBR'] = testData['SUBCD_NBR'].astype(float)
    testData['PT_AGE'] = testData['PT_AGE'].astype(int)
    testData['PT_ZIP'] = testData['PT_ZIP'].astype(float)
    metadata = ["MBR_ID", "FST_NAME", "MDL_NAME", 'LST_NAME','ICD_CD','CPT_CD']
    # Get the training data length 
    #NData = x_PT_AGE.shape[0]
    
    #x_tst = get_tensor_dataset(x_PT_AGE,x_pt_zip,x_icd_cd,x_risk_strat,x_sub_cdbr)       
    # Retrieve the latest version from the registry
    model = retrieve_model("XGBRegressor_clmsPredmodel_0171")
    if model is not None:
        testData_pandas = testData.to_pandas()
            
        predictions_df = model.run(
            testData_pandas,
            function_name="predict"
        )
        print(predictions_df.columns)
        # Write DataFrame to a permanent Snowflake table
        predictions_df.drop(columns=["ICDCD_NUMCODED", "SUBCD_NBR"], inplace=True)
        session.write_pandas(
            df=predictions_df, 
            table_name=f"{database_name}.{schema_name}.ClmCostPredictions_{int(last_yr)+1}", 
            auto_create_table=True,  # Automatically creates table if missing
            overwrite=True           # Overwrites contents if table exists
        )
        print("Claim cost predictions written to table")
        """
        predictions_df.to_snowflake(
            name=f"{database_name}.{schema_name}.ClmCostPredictions_{last_yr}", 
            if_exists="replace",  # Options: 'fail', 'replace', 'append'
            index=False           # Set to True if you want to keep the DataFrame index
        )
        #predictions_df.write.mode("overwrite").save_as_table(f"{database_name}.{schema_name}.ClmCostPredictions_{last_yr}")
        """
    else:
        print("Trained Model not available")


In [ ]:
test()
